In [3]:
%%file probny.py

from kafka import KafkaConsumer
from collections import deque
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
'smart_home',
bootstrap_servers='broker:9092',
auto_offset_reset='earliest',
group_id='anomaly-group',
value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)


windows = {
"motion_sensor": deque(),
"fridge": deque(),
"oven": deque(),
"stove": deque(),
"tv": deque(),
"air_conditioner": deque(),
"washing_machine": deque(),
"light": deque()
}

window_sizes = {
"motion_sensor": timedelta(minutes=5),
"fridge": timedelta(minutes=5),
"light": timedelta(minutes=30),
"oven": timedelta(minutes=30),
"stove": timedelta(minutes=30),
"tv": timedelta(hours=4),
"air_conditioner": timedelta(hours=2),
"washing_machine": timedelta(hours=3)
}

msg_count = 0

print("Uruchomiono konsumenta anomalii...")

for message in consumer:

    event = message.value
    msg_count += 1

    device = event["device_type"]
    now = datetime.fromisoformat(event["timestamp"])

    windows[device].append(event)

    cutoff = now - window_sizes[device]

    while windows[device]:

        oldest_time = datetime.fromisoformat(
            windows[device][0]["timestamp"]
        )

        if oldest_time < cutoff:
            windows[device].popleft()
        else:
            break

    if device == "fridge":

        if (
            event["door_status"] == "OPEN"
            and event["open_duration_sec"] > 120
        ):
            print(
                f"[ANOMALIA] Lodówka otwarta "
                f"{event['open_duration_sec']} sekund."
            )

        avg_temp = (
            sum(e["temperature"] for e in windows["fridge"])
            / len(windows["fridge"])
        )

        if avg_temp > 7:
            print(
                f"[ANOMALIA] Średnia temperatura lodówki "
                f"w ostatnich 5 min = {avg_temp:.1f}°C"
            )

    elif device == "oven":

        if (
            event["status"] == "ON"
            and event["working_time_min"] > 90
        ):
            print(
                f"[ANOMALIA] Piekarnik działa "
                f"{event['working_time_min']} minut."
            )

    elif device == "stove":

        if (
            event["status"] == "ON"
            and event["working_time_min"] > 60
        ):
            print(
                f"[ANOMALIA] Kuchenka działa "
                f"{event['working_time_min']} minut."
            )

    elif device == "tv":

        if (
            event["status"] == "ON"
            and event["watch_time_min"] > 240
        ):
            print(
                f"[ANOMALIA] TV działa "
                f"{event['watch_time_min']} minut."
            )

    elif device == "air_conditioner":

        if (
            event["status"] == "ON"
            and event["working_time_min"] > 360
        ):
            print(
                f"[ANOMALIA] Klimatyzacja działa "
                f"{event['working_time_min']} minut."
            )

    elif device == "washing_machine":

        if (
            event["status"] == "RUNNING"
            and event["remaining_time_min"] > 150
        ):
            print(
                f"[ANOMALIA] Długi program prania "
                f"({event['remaining_time_min']} min)."
            )

    elif device == "light":

        recent_motion = len(windows["motion_sensor"])

        if (
            event["status"] == "ON"
            and recent_motion == 0
        ):
            print(
                f"[ANOMALIA] Światło włączone, "
                f"ale brak ruchu w ostatnich 5 minutach."
            )

    if msg_count % 50 == 0:
        print(f"\nPrzetworzono {msg_count} komunikatów.\n")


Overwriting probny.py
